In [ ]:
import os
import glob
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
import matplotlib.pyplot as plt

print(f"TensorFlow Version: {tf.__version__}")
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

# Ensure directories exist based on the project structure
SAVED_MODELS_DIR = '../saved_models/'
os.makedirs(SAVED_MODELS_DIR, exist_ok=True)

In [ ]:
import os
import glob
import tensorflow as tf

# Path alignments for the local dataset
TRAIN_IMG_DIR = '../data/train/images/'
TRAIN_MASK_DIR = '../data/train/labels/' # <-- Updated folder name

IMG_SIZE = (256, 256)
BATCH_SIZE = 16

# Grab all file paths
image_paths = sorted(glob.glob(os.path.join(TRAIN_IMG_DIR, '*.*')))
mask_paths = sorted(glob.glob(os.path.join(TRAIN_MASK_DIR, '*.*')))

print(f"DEBUG: Found {len(image_paths)} images and {len(mask_paths)} masks.")

def load_and_preprocess(img_path, mask_path):
    """Loads and standardizes SAR images and binary masks."""
    # Process image
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3) 
    img = tf.image.resize(img, IMG_SIZE) / 255.0
    
    # Process mask
    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(mask, channels=1) 
    mask = tf.image.resize(mask, IMG_SIZE) / 255.0
    mask = tf.math.round(mask)
    
    return img, mask

if len(image_paths) == 0 or len(image_paths) != len(mask_paths):
    print("🚨 FATAL ERROR: Mismatch or no files found. Check your directories!")
else:
    # Build the tf.data.Dataset pipeline
    dataset = tf.data.Dataset.from_tensor_slices((image_paths, mask_paths))
    dataset = dataset.shuffle(buffer_size=len(image_paths), seed=42)
    dataset = dataset.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)

    # 80/20 Train/Validation Split
    train_size = int(0.8 * len(image_paths))
    train_dataset = dataset.take(train_size).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    val_dataset = dataset.skip(train_size).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    # Safely calculate batches
    train_batches = tf.data.experimental.cardinality(train_dataset).numpy()
    val_batches = tf.data.experimental.cardinality(val_dataset).numpy()

    print(f"✅ Pipeline ready! Training batches: {train_batches}, Validation batches: {val_batches}")

In [ ]:
def sparse_conv_block(x, filters, groups=4):
    """
    Modern Sparse Block: Replaces heavy dense convolutions.
    1. SeparableConv2D handles spatial sparsity.
    2. Grouped Conv2D handles cross-channel sparsity.
    3. L1 Regularization drops near-zero weights to enforce actual sparsity.
    """
    # 1. Spatial Sparsity Step
    x = layers.SeparableConv2D(
        filters, 
        kernel_size=3, 
        padding='same', 
        use_bias=False,
        kernel_regularizer=regularizers.l1(1e-5)
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    # 2. Channel Sparsity Step
    # Grouped convolutions split the channels into smaller sparse blocks
    x = layers.Conv2D(
        filters, 
        kernel_size=1, 
        groups=groups, 
        padding='same', 
        use_bias=False,
        kernel_regularizer=regularizers.l1(1e-5)
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    return x

def build_sparse_seg_model(input_shape=(256, 256, 3), num_classes=1):
    inputs = layers.Input(shape=input_shape)
    
    # Sparse Encoder
    e1 = sparse_conv_block(inputs, 32, groups=4)
    p1 = layers.MaxPooling2D(2)(e1)
    
    e2 = sparse_conv_block(p1, 64, groups=4)
    p2 = layers.MaxPooling2D(2)(e2)
    
    e3 = sparse_conv_block(p2, 128, groups=4)
    p3 = layers.MaxPooling2D(2)(e3)
    
    # Sparse Bottleneck
    b = sparse_conv_block(p3, 256, groups=8)
    
    # Sparse Decoder
    d1 = layers.UpSampling2D(2)(b)
    d1 = layers.Concatenate()([d1, e3])
    d1 = sparse_conv_block(d1, 128, groups=4)
    
    d2 = layers.UpSampling2D(2)(d1)
    d2 = layers.Concatenate()([d2, e2])
    d2 = sparse_conv_block(d2, 64, groups=4)
    
    d3 = layers.UpSampling2D(2)(d2)
    d3 = layers.Concatenate()([d3, e1])
    d3 = sparse_conv_block(d3, 32, groups=4)
    
    outputs = layers.Conv2D(num_classes, 1, activation='sigmoid')(d3)
    
    model = models.Model(inputs, outputs, name='Sparse_Oil_Spill_Model')
    return model

# Instantiate and verify parameters
sparse_model = build_sparse_seg_model()
sparse_model.summary()

In [ ]:
# Compile the sparse network
sparse_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.MeanIoU(num_classes=2)]
)

# Callbacks for saving to the repository's expected folder
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        os.path.join(SAVED_MODELS_DIR, 'sparse_oil_spill.h5'),
        save_best_only=True,
        monitor='val_loss'
    ),
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)
]

# Start the training!
print("Starting model training...")
history = sparse_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=30,
    callbacks=callbacks
)

In [ ]:
def visualize_predictions(image_batch, true_masks, model):
    """Generates predictions and plots them against SAR inputs."""
    predictions = model.predict(image_batch)
    
    # Establish operational thresholds
    detection_sensitivity = 0.05
    predicted_masks = (predictions > detection_sensitivity).astype(np.uint8)
    
    fig, axes = plt.subplots(BATCH_SIZE, 3, figsize=(15, 5 * BATCH_SIZE))
    for i in range(BATCH_SIZE):
        # Original Image
        axes[i, 0].imshow(image_batch[i])
        axes[i, 0].set_title("SAR Image")
        axes[i, 0].axis('off')
        
        # Ground Truth Mask
        axes[i, 1].imshow(true_masks[i].numpy().squeeze(), cmap='gray')
        axes[i, 1].set_title("Ground Truth")
        axes[i, 1].axis('off')
        
        # Sparse Model Prediction
        axes[i, 2].imshow(predicted_masks[i].squeeze(), cmap='gray')
        axes[i, 2].set_title(f"Sparse Prediction (Sens: {detection_sensitivity})")
        axes[i, 2].axis('off')
        
    plt.tight_layout()
    plt.show()

# Visualize a batch
# sample_images, sample_masks = next(iter(test_dataset))
# visualize_predictions(sample_images, sample_masks, sparse_model)